# 10 · End-to-End Pipeline

> **Module 3 · Case Studies · Lecture 3.3**

In this notebook you will:

- Lay out a bronze / silver / gold medallion architecture
- Execute the pipeline stage by stage, logging row counts at every step
- Demonstrate an idempotent re-run — run it twice and get identical results
- Implement a quality gate that raises when a threshold is breached
- Load the gold layer to BigQuery and submit the same logic as a Dataproc job

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · The medallion architecture

```
  raw source            BRONZE                SILVER                    GOLD
  -----------         ---------------   ----------------------   -----------------
  POS transactions  ->  raw, as-is      ->  deduped, typed,     ->  daily aggregates
  (CSV / JSON)         + ingest_ts        enriched with dims        ready for BI

  landing zone          immutable        conformed, curated      business-facing
  (append-only)         source of truth  quality-checked          (BigQuery tables)
```

| Layer | Contains | Written by | Consumers |
|---|---|---|---|
| **Bronze** | Raw data exactly as received, plus `ingest_ts` and a `source_file` marker | Ingestion job | Silver |
| **Silver** | Deduplicated, typed, enriched, validated rows | Transformation job | Silver, Gold, analysts |
| **Gold** | Business aggregates at reporting grain | Aggregation job | Dashboards, BigQuery |

**Note:** Each layer is a separate contract. A schema change in Bronze is absorbed in Silver
and never reaches Gold.

**Caution:** Bronze is **append-only**. Never update or delete a Bronze row — that destroys the
audit trail and makes replay impossible.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


## 2 · Generate the source data (bronze)

We generate a deterministic POS feed and write it to a local "landing zone" as CSV —
the same shape the pipeline would find in `gs://bucket/landing/`.

The source contains three kinds of problems the pipeline must handle:

1. **Duplicate transactions** (the store network re-sends them).
2. **Dirty numerics** — `"not_a_number"` and a blank amount.
3. **Missing dimension rows** — some customers are not in the customer dimension.

In [ ]:
import json
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

PIPELINE_ROOT = "/tmp/dataproc_notebook_10_pipeline"
if os.path.exists(PIPELINE_ROOT):
    shutil.rmtree(PIPELINE_ROOT)

LANDING = f"{PIPELINE_ROOT}/landing"
BRONZE = f"{PIPELINE_ROOT}/bronze"
SILVER = f"{PIPELINE_ROOT}/silver"
GOLD = f"{PIPELINE_ROOT}/gold"
for path in (LANDING, BRONZE, SILVER, GOLD):
    os.makedirs(path, exist_ok=True)

SEED = 2026
N_ROWS = 3000

# ------------------------------------------------------------------
# The raw source: strings, duplicates and dirt included.
# ------------------------------------------------------------------
raw_source = (
    spark.range(N_ROWS)
    .withColumn("transaction_id", F.concat(F.lit("tx-"), F.col("id").cast("string")))
    .withColumn("customer_id", F.concat(F.lit("cust-"), (F.col("id") % 60).cast("string")))
    .withColumn("product_id", F.concat(F.lit("prod-"), (F.col("id") % 15).cast("string")))
    .withColumn("store_id", F.concat(F.lit("store-"), (F.col("id") % 10).cast("string")))
    .withColumn("quantity", F.col("id").cast("string"))
    .withColumn("unit_price", F.round(F.rand(SEED) * 150 + 5, 2).cast("string"))
    .withColumn("sale_date", F.date_add(F.lit("2026-09-01").cast("date"),
                                        (F.col("id") % 45).cast("int")).cast("string"))
)

# Dirty rows: every 500th transaction has an unusable amount.
raw_source = raw_source.withColumn(
    "unit_price",
    F.when(F.col("id") % 500 == 0, F.lit("not_a_number"))
     .when(F.col("id") % 700 == 0, F.lit(""))
     .otherwise(F.col("unit_price")),
)

# Write two landing files, as a real feed would arrive.
landing_files = []
for part in range(2):
    path = f"{LANDING}/part-{part:05d}.csv"
    (raw_source.filter((F.col("id") % 2) == part)
     .drop("id")
     .write.mode("overwrite").option("header", True).csv(path))
    landing_files.append(path)

print("landing files:")
for path in landing_files:
    print("  ", path, os.path.getsize(path), "bytes")

# Dimensions (also "landed" separately).
customers = (
    spark.range(50)
    .withColumn("customer_id", F.concat(F.lit("cust-"), F.col("id").cast("string")))
    .withColumn("customer_name", F.concat(F.lit("Customer "), F.col("id").cast("string")))
    .withColumn("loyalty_tier", F.when(F.col("id") % 10 == 0, F.lit("gold"))
                                    .when(F.col("id") % 3 == 0, F.lit("silver"))
                                    .otherwise(F.lit("bronze")))
    .drop("id")   # drop the surrogate key, or the join produces two `id` columns
)
products = (
    spark.range(15)
    .withColumn("product_id", F.concat(F.lit("prod-"), F.col("id").cast("string")))
    .withColumn("product_name", F.concat(F.lit("Product "), F.col("id").cast("string")))
    .withColumn("category", F.concat(F.lit("Category "), (F.col("id") % 5).cast("string")))
    .drop("id")
)

print("landing dir:", LANDING)

## 3 · Stage 1 — Bronze (raw landing)

Bronze is a straight copy of the source with two additions: an `ingest_ts` timestamp
and a `source_file` marker so you can trace any row back to its origin.

Bronze is written with `mode("overwrite")` on a **per-partition** basis so a re-run
replaces the partition rather than appending a second copy.

In [ ]:
from datetime import datetime, timezone

def log_stage(stage, count, extra=""):
    """Log a stage boundary with its row count."""
    print(f"[{datetime.now(timezone.utc).isoformat(timespec='seconds')}] "
          f"{stage:<28s} rows={count:<8d} {extra}")

bronze_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id",    StringType(), True),
    StructField("product_id",     StringType(), True),
    StructField("store_id",       StringType(), True),
    StructField("quantity",       StringType(), True),
    StructField("unit_price",     StringType(), True),
    StructField("sale_date",      StringType(), True),
])

bronze = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(bronze_schema)
    .csv(f"{LANDING}/part-*.csv")
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

bronze.write.mode("overwrite").parquet(f"{BRONZE}/transactions")
bronze_count = spark.read.parquet(f"{BRONZE}/transactions").count()
log_stage("BRONZE (raw landing)", bronze_count)

spark.read.parquet(f"{BRONZE}/transactions").show(5, truncate=False)

## 4 · Stage 2 — Silver (conformed and enriched)

Silver applies four transformations:

1. **Dedupe** — keep the latest ingest per `transaction_id` (deterministic).
2. **Type** — parse the strings with `try_cast` so a bad value becomes null, not an
   exception.
3. **Validate** — reject rows that fail the contract, and route them to quarantine.
4. **Enrich** — join the dimensions with `F.broadcast`.

In [ ]:
from pyspark.sql import Window

# --- 4a. Dedupe ---------------------------------------------------------
dedupe_window = Window.partitionBy("transaction_id").orderBy(
    F.col("ingest_ts").desc(), F.col("transaction_id").desc()
)

deduped = (
    bronze
    .withColumn("rn", F.row_number().over(dedupe_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
log_stage("SILVER dedup", deduped.count(),
          f"(removed {bronze_count - deduped.count()} duplicate rows)")

# --- 4b. Type with try_cast --------------------------------------------
typed = (
    deduped
    .withColumn("quantity", F.expr("try_cast(quantity AS int)"))
    .withColumn("unit_price", F.expr("try_cast(unit_price AS decimal(18,2))"))
    .withColumn("sale_date", F.expr("try_cast(sale_date AS date)"))
)
typed.printSchema()

In [ ]:
# --- 4c. Validate and quarantine --------------------------------------
typed = typed.withColumn(
    "rejection_reason",
    F.when(F.col("transaction_id").isNull(), F.lit("MISSING_TRANSACTION_ID"))
     .when(F.col("customer_id").isNull(), F.lit("MISSING_CUSTOMER_ID"))
     .when(F.col("sale_date").isNull(), F.lit("MISSING_SALE_DATE"))
     .when(F.col("quantity").isNull(), F.lit("MISSING_QUANTITY"))
     .when(F.col("quantity") <= 0, F.lit("NON_POSITIVE_QUANTITY"))
     .when(F.col("unit_price").isNull(), F.lit("INVALID_UNIT_PRICE"))
     .when(F.col("unit_price") <= 0, F.lit("NON_POSITIVE_PRICE"))
     .otherwise(F.lit(None)),
)

good = typed.filter(F.col("rejection_reason").isNull()).drop("rejection_reason")
bad = typed.filter(F.col("rejection_reason").isNotNull())

log_stage("SILVER valid", good.count())
log_stage("SILVER quarantine", bad.count())

bad.select("transaction_id", "rejection_reason", "source_file") \
   .write.mode("overwrite").json(f"{PIPELINE_ROOT}/quarantine/transactions")

bad.groupBy("rejection_reason").count().orderBy(F.desc("count")).show(truncate=False)

In [ ]:
# --- 4d. Enrich with broadcast dimension joins ------------------------
silver = (
    good
    .join(F.broadcast(customers), "customer_id", "left")
    .join(F.broadcast(products), "product_id", "left")
    .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("net_amount", F.col("gross_amount"))
    .withColumn("dimension_status",
                F.when(F.col("customer_name").isNull(), F.lit("MISSING_CUSTOMER"))
                 .when(F.col("product_name").isNull(), F.lit("MISSING_PRODUCT"))
                 .otherwise(F.lit("COMPLETE")))
)

silver.write.mode("overwrite").option("compression", "snappy").parquet(f"{SILVER}/transactions")
silver_count = spark.read.parquet(f"{SILVER}/transactions").count()
log_stage("SILVER (conformed)", silver_count)

silver.select(
    "transaction_id", "customer_name", "product_name", "category",
    "quantity", "unit_price", "gross_amount", "dimension_status",
).show(5, truncate=False)

print("\ndimension completeness:")
silver.groupBy("dimension_status").count().show()

## 5 · Stage 3 — Gold (business aggregates)

Gold is the reporting layer: one row per `sale_date × category × loyalty_tier`, with
the measures the dashboard needs.

In [ ]:
gold = (
    silver
    .groupBy(
        F.col("sale_date").alias("sale_date"),
        F.col("category").alias("category"),
        F.col("loyalty_tier").alias("loyalty_tier"),
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("transaction_id").alias("distinct_transactions"),
        F.countDistinct("customer_id").alias("distinct_customers"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("gross_amount"), 2).alias("gross_revenue"),
        F.round(F.avg("gross_amount"), 2).alias("avg_order_value"),
    )
    .orderBy("sale_date", "category", "loyalty_tier")
)

gold.write.mode("overwrite").option("compression", "snappy").parquet(f"{GOLD}/daily_sales")
gold_count = spark.read.parquet(f"{GOLD}/daily_sales").count()
log_stage("GOLD (daily summary)", gold_count)

gold.show(15, truncate=False)

## 6 · The quality gate

A quality gate **raises** when a threshold is breached. Logging a warning is not a gate —
a gate must stop the pipeline.

| Gate | Threshold | Why |
|---|---|---|
| Bronze row count | > 0 | An empty landing zone usually means a broken upstream. |
| Duplicate rate | < 5% | A sudden spike means the source is misbehaving. |
| Quarantine rate | < 2% | More than 2% invalid means something is structurally wrong. |
| Silver reconciliation | `gross == quantity * price` | Catches a bug in the arithmetic. |
| Gold row count | > 0 and <= Silver rows | The aggregate must not invent or lose rows. |

In [ ]:
class QualityGateError(RuntimeError):
    """Raised when a pipeline quality gate is breached."""


def run_quality_gates(stage_counts, silver_df, gold_df, quarantine_count, source_rows):
    """Raise QualityGateError if any gate is breached."""
    failures = []

    bronze_rows = stage_counts["bronze"]
    silver_rows = stage_counts["silver"]
    gold_rows = stage_counts["gold"]

    # Gate 1: bronze must not be empty.
    if bronze_rows <= 0:
        failures.append(f"bronze is empty ({bronze_rows} rows)")

    # Gate 2: duplicate rate below 5%.
    duplicate_rate = (bronze_rows - stage_counts["deduped"]) / bronze_rows if bronze_rows else 1.0
    if duplicate_rate >= 0.05:
        failures.append(f"duplicate rate {duplicate_rate:.2%} >= 5%")

    # Gate 3: quarantine rate below 2%.
    quarantine_rate = quarantine_count / bronze_rows if bronze_rows else 1.0
    if quarantine_rate >= 0.02:
        failures.append(f"quarantine rate {quarantine_rate:.2%} >= 2%")

    # Gate 4: silver arithmetic reconciles.
    mismatches = silver_df.filter(
        F.abs(F.col("gross_amount") - (F.col("quantity") * F.col("unit_price"))) > 0.01
    ).count()
    if mismatches > 0:
        failures.append(f"{mismatches} silver rows do not reconcile")

    # Gate 5: gold is a strict, non-empty aggregation of silver.
    if gold_rows <= 0:
        failures.append("gold is empty")
    if gold_rows > silver_rows:
        failures.append(f"gold rows ({gold_rows}) > silver rows ({silver_rows})")

    print("  quality gates:")
    print(f"    bronze rows         : {bronze_rows}")
    print(f"    duplicate rate      : {duplicate_rate:.2%}  (limit 5%)")
    print(f"    quarantine rate     : {quarantine_rate:.2%}  (limit 2%)")
    print(f"    reconciliation fails: {mismatches}")
    print(f"    gold rows           : {gold_rows}")

    if failures:
        for failure in failures:
            print(f"    [FAIL] {failure}")
        raise QualityGateError("Quality gate failed: " + "; ".join(failures))

    print("    [PASS] all quality gates satisfied")
    return True


stage_counts = {
    "bronze": bronze_count,
    "deduped": deduped.count(),
    "silver": silver_count,
    "gold": gold_count,
}

silver_df = spark.read.parquet(f"{SILVER}/transactions")
gold_df = spark.read.parquet(f"{GOLD}/daily_sales")

run_quality_gates(stage_counts, silver_df, gold_df, bad.count(), bronze_count)

### Proving the gate actually fails

A gate that never fires is not a gate. Feed it a deliberately broken input and watch it
raise.

In [ ]:
try:
    run_quality_gates(
        stage_counts={"bronze": 0, "deduped": 0, "silver": 0, "gold": 0},
        silver_df=silver_df,
        gold_df=gold_df,
        quarantine_count=0,
        source_rows=0,
    )
except QualityGateError as exc:
    print("\nThe gate raised as designed:")
    print("   ", exc)

## 7 · Idempotent re-run

The pipeline must produce **identical** results when it runs twice. Two properties give
us that:

1. Every stage writes with `mode("overwrite")` to a deterministic path.
2. The source data is fixed (a deterministic seed), so the input does not change.

We run the full pipeline twice and compare the row counts and a content signature of
the gold layer.

In [ ]:
def gold_signature(df):
    """A deterministic content signature of the gold layer."""
    row = df.agg(
        F.count("*").alias("rows"),
        F.round(F.sum("gross_revenue"), 2).alias("revenue"),
        F.sum("transaction_count").alias("txns"),
        F.sum("units_sold").alias("units"),
    ).collect()[0]
    return (row["rows"], row["revenue"], row["txns"], row["units"])


def run_pipeline():
    """The full pipeline, stage by stage, with row counts logged."""
    counts = {}

    # Bronze
    b = (
        spark.read
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .schema(bronze_schema)
        .csv(f"{LANDING}/part-*.csv")
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.input_file_name())
    )
    b.write.mode("overwrite").parquet(f"{BRONZE}/transactions")
    counts["bronze"] = spark.read.parquet(f"{BRONZE}/transactions").count()

    # Silver
    d = (
        spark.read.parquet(f"{BRONZE}/transactions")
        .withColumn("rn", F.row_number().over(dedupe_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )
    counts["deduped"] = d.count()

    t = (
        d
        .withColumn("quantity", F.expr("try_cast(quantity AS int)"))
        .withColumn("unit_price", F.expr("try_cast(unit_price AS decimal(18,2))"))
        .withColumn("sale_date", F.expr("try_cast(sale_date AS date)"))
        .withColumn(
            "rejection_reason",
            F.when(F.col("transaction_id").isNull(), F.lit("MISSING_TRANSACTION_ID"))
             .when(F.col("customer_id").isNull(), F.lit("MISSING_CUSTOMER_ID"))
             .when(F.col("sale_date").isNull(), F.lit("MISSING_SALE_DATE"))
             .when(F.col("quantity").isNull(), F.lit("MISSING_QUANTITY"))
             .when(F.col("quantity") <= 0, F.lit("NON_POSITIVE_QUANTITY"))
             .when(F.col("unit_price").isNull(), F.lit("INVALID_UNIT_PRICE"))
             .when(F.col("unit_price") <= 0, F.lit("NON_POSITIVE_PRICE"))
             .otherwise(F.lit(None)),
        )
    )
    g = t.filter(F.col("rejection_reason").isNull()).drop("rejection_reason")
    counts["quarantine"] = t.count() - g.count()

    s = (
        g
        .join(F.broadcast(customers), "customer_id", "left")
        .join(F.broadcast(products), "product_id", "left")
        .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
        .withColumn("net_amount", F.col("gross_amount"))
        .withColumn("dimension_status",
                    F.when(F.col("customer_name").isNull(), F.lit("MISSING_CUSTOMER"))
                     .when(F.col("product_name").isNull(), F.lit("MISSING_PRODUCT"))
                     .otherwise(F.lit("COMPLETE")))
    )
    s.write.mode("overwrite").option("compression", "snappy").parquet(f"{SILVER}/transactions")
    counts["silver"] = spark.read.parquet(f"{SILVER}/transactions").count()

    # Gold
    gd = (
        spark.read.parquet(f"{SILVER}/transactions")
        .groupBy(
            F.col("sale_date").alias("sale_date"),
            F.col("category").alias("category"),
            F.col("loyalty_tier").alias("loyalty_tier"),
        )
        .agg(
            F.count("*").alias("transaction_count"),
            F.countDistinct("transaction_id").alias("distinct_transactions"),
            F.countDistinct("customer_id").alias("distinct_customers"),
            F.sum("quantity").alias("units_sold"),
            F.round(F.sum("gross_amount"), 2).alias("gross_revenue"),
            F.round(F.avg("gross_amount"), 2).alias("avg_order_value"),
        )
    )
    gd.write.mode("overwrite").option("compression", "snappy").parquet(f"{GOLD}/daily_sales")
    counts["gold"] = spark.read.parquet(f"{GOLD}/daily_sales").count()

    for stage in ("bronze", "deduped", "quarantine", "silver", "gold"):
        log_stage(f"RUN {stage}", counts[stage])

    return counts


print("== RUN 1 ==")
run_1_counts = run_pipeline()
run_1_sig = gold_signature(spark.read.parquet(f"{GOLD}/daily_sales"))
print("gold signature (rows, revenue, txns, units):", run_1_sig)

print("\n== RUN 2 ==")
run_2_counts = run_pipeline()
run_2_sig = gold_signature(spark.read.parquet(f"{GOLD}/daily_sales"))
print("gold signature (rows, revenue, txns, units):", run_2_sig)

print("\n== IDEMPOTENCY CHECK ==")
print("stage counts identical :", run_1_counts == run_2_counts)
print("gold signature identical:", run_1_sig == run_2_sig)
print("RESULT:", "IDEMPOTENT" if (run_1_counts == run_2_counts and run_1_sig == run_2_sig)
      else "NOT IDEMPOTENT")

## 8 · Load the gold layer to BigQuery

The final step publishes the gold aggregates to BigQuery, where dashboards read them.

**Caution:** BigQuery writes require a `temporaryGcsBucket`, and `writeDisposition` must be
chosen deliberately — `WRITE_TRUNCATE` replaces the table, `WRITE_APPEND` adds to it.

### Optional: BigQuery load

Uncomment after `gcloud auth application-default login` and creating the dataset.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — load the gold layer to BigQuery.
# ------------------------------------------------------------------
# BQ_PROJECT = "your-project-id"
# BQ_DATASET = "analytics"
# BQ_TABLE = "daily_sales"
#
# gold_final = spark.read.parquet(f"{GOLD}/daily_sales")
# (
#     gold_final.write
#     .format("bigquery")
#     .option("writeDisposition", "WRITE_TRUNCATE")
#     .option("temporaryGcsBucket", "your-staging-bucket")
#     .option("writeMethod", "direct")
#     .save(f"{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}")
# )
#
# # Read it back to verify.
# verify = (
#     spark.read
#     .format("bigquery")
#     .option("query", f"SELECT COUNT(*) AS n FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`")
#     .load()
# )
# verify.show()

print("BigQuery load cell is commented out — no GCP account required for this notebook.")

## 9 · Run the same logic as a Dataproc job

The notebook calls functions; a Dataproc job calls the **same functions** from a script.

### The script

```python
# jobs/pipeline.py
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# NOTE: no master="local[*]" — the cluster provides it.
spark = SparkSession.builder.appName("retail-pipeline").getOrCreate()
...
```

### Submit it

```bash
# Ship the script and any shared module to GCS.
gcloud storage cp jobs/pipeline.py gs://YOUR_BUCKET/jobs/
gcloud storage cp jobs/common.py   gs://YOUR_BUCKET/jobs/

gcloud dataproc jobs submit pyspark gs://YOUR_BUCKET/jobs/pipeline.py \
  --project=YOUR_PROJECT_ID \
  --region=us-central1 \
  --cluster=your-dataproc-cluster \
  --py-files=gs://YOUR_BUCKET/jobs/common.py \
  --driver-memory=4g \
  --executor-memory-overhead=2g \
  --properties=spark.sql.shuffle.partitions=200,spark.sql.adaptive.enabled=true
```

### The same pipeline as a local `spark-submit`

```bash
spark-submit \
  --master local[4] \
  --conf spark.sql.shuffle.partitions=8 \
  --py-files common.py \
  pipeline.py
```

### Read the job result

```bash
JOB_ID=$(gcloud dataproc jobs list --region=us-central1 --limit=1 --format="value(jobId)")
gcloud dataproc jobs wait "$JOB_ID" --region=us-central1
gcloud dataproc jobs describe "$JOB_ID" --region=us-central1
```

**Caution:** **`--py-files` is not optional.** Workers do not have your laptop's filesystem. Any
module you `import` must be shipped with the job.

**Caution:** **Never hardcode `master`** in a submitted script — it makes the whole job run on the
driver node.

In [ ]:
# The portable session pattern, one last time.
import sys

def build_session(app_name):
    builder = SparkSession.builder.appName(app_name)
    if "spark-submit" not in sys.argv[0] and "notebook" not in sys.argv[0]:
        builder = builder.master("local[*]")
    return builder.getOrCreate()

# On Dataproc: build_session("retail-pipeline")  -> uses the cluster's yarn master.
print("This session's master:", spark.sparkContext.master)
print("Application id        :", spark.sparkContext.applicationId)
print("Parallelism           :", spark.sparkContext.defaultParallelism)

## 10 · Pipeline checklist

- [ ] **Bronze is append-only** and records `ingest_ts` plus `source_file`.
- [ ] Every stage writes with an explicit mode and an explicit schema.
- [ ] Every stage logs its row count.
- [ ] Deduplication is deterministic (`row_number` with a tiebreaker).
- [ ] Types are parsed with `try_cast` at the boundary; downstream uses plain types.
- [ ] Invalid rows go to **quarantine with a reason**, never dropped.
- [ ] Dimension joins use `F.broadcast` when the dimension is small.
- [ ] A **quality gate raises** on threshold breaches — it does not just log.
- [ ] The pipeline is idempotent: re-running produces identical results.
- [ ] The gold layer is loaded to BigQuery with an explicit `writeDisposition`.
- [ ] The submitted script omits `master` and ships its modules with `--py-files`.

## 🔑 Key Takeaways

- The medallion layout — bronze (raw, append-only), silver (conformed, enriched), gold (aggregated) — gives each layer a separate contract.
- Log the row count at **every** stage boundary; a pipeline you cannot count is a pipeline you cannot trust.
- Bronze records `ingest_ts` and `source_file` so every row is traceable back to its origin.
- Deduplicate deterministically with `row_number` plus a tiebreaker; parse types with `try_cast` at the boundary.
- **Quarantine invalid rows with a reason** — never drop them silently.
- A quality gate must **raise**, not warn; a gate that never fires is not a gate.
- Idempotency comes from deterministic inputs plus `mode("overwrite")` on every stage — verify by running twice.
- BigQuery writes need a `temporaryGcsBucket` and an explicit `writeDisposition`.
- On Dataproc, ship shared modules with `--py-files` and never hardcode `master`.

---

## 📚 Next steps

- Move the pipeline functions into `jobs/pipeline.py` and submit it with `gcloud dataproc jobs submit pyspark`.
- Add a silver-to-gold reconciliation gate that compares the sum of the daily aggregates with the silver totals.
- Extend the pipeline with a second source (a BigQuery dimension table) and align the schemas with `unionByName`.
